Working memory training: Second level GLM analysis of working memory training fMRI data<br>
Second-level GLM analysis of activation patterns

Preprocessing via fMRIPrep: coregistration, normalization, unwarping, noise component extraction, segmentation, skullstripping, etc.<br>
Denoising:<br>
Confounds:<br>
* 6 motion params (3 translations + 3 rotations)
* 6 sCompCor params (PCA based)
* std_dvars
* framewise_displacement

hrf_model='glover' <br>
drift_model='cosine' <br>
high_pass=0.008 <br>
noise_model='ar1' <br>
<br>
no low pass <br>
As a mask I used 
<br><br>


First level contrasts <br>
5 numbers = 5 conditions <br>
'0back-1back': np.pad([1, -1, 0, 0, 0], (0,n_columns-5)), <br>
                    '0back-2back': np.pad([1, 0, -1, 0, 0], (0,n_columns-5)), <br>
                    '0back-4back': np.pad([1, 0, 0, -1, 0], (0,n_columns-5)), <br>
                    '0back-fix': np.pad([1, 0, 0, 0, -1], (0,n_columns-5)), <br>
                    '1back-0back': np.pad([-1, 1, 0, 0, 0], (0,n_columns-5)), <br>
                    '1back-2back': np.pad([0, 1, -1, 0, 0], (0,n_columns-5)), <br>
                    '1back-4back': np.pad([0, 1, 0, -1, 0], (0,n_columns-5)), <br>
                    '1back-fix': np.pad([0, 1, 0, 0, -1], (0,n_columns-5)), <br>
                    '2back-0back': np.pad([-1, 0, 1, 0, 0], (0,n_columns-5)), <br>
                    '2back-1back': np.pad([0, -1, 1, 0, 0], (0,n_columns-5)), <br>
                    '2back-4back': np.pad([0, 0, 1, -1, 0], (0,n_columns-5)), <br>
                    '2back-fix': np.pad([0, 0, 1, 0, -1], (0,n_columns-5)), <br>
                    '4back-0back': np.pad([-1, 0, 0, 1, 0], (0,n_columns-5)), <br>
                    '4back-1back': np.pad([0, -1, 0, 1, 0], (0,n_columns-5)), <br>
                    '4back-2back': np.pad([0, 0, -1, 1, 0], (0,n_columns-5)),  <br>
                    '4back-fix': np.pad([0, 0, 0, 1, -1], (0,n_columns-5)), <br>
                    '0back': np.pad([1, 0, 0, 0, 0], (0,n_columns-5)), <br>
                    '1back': np.pad([0, 1, 0, 0, 0], (0,n_columns-5)), <br>
                    '2back': np.pad([0, 0, 1, 0, 0], (0,n_columns-5)), <br>
'4back': np.pad([0, 0, 0, 1, 0], (0,n_columns-5)), <br>
'fix': np.pad([0, 0, 0, 0, 1], (0,n_columns-5)), <br>
'effects_of_interest': np.eye(n_columns)[:5] <br>

Still possible to try on the first level: no denoising, ICA-based denoising. Add low-pass filter. <br>

Do smoothing on this level <br>
Second level analysis:
* nilearn: specify 2nd level contrast: <br>
    ** 'ses+control-sport': np.pad([0, -1, 1, 1, -1, 0], (0,n_columns-6)), and so on <br>
    ** ['control', 'sport', 'med'] basically each group against 2 remaining
    compute each of these contrasts for 3 types of 1st level contrasts <br>
* t-test for each group pre-post and then permutation test to correct for multiple comparisons
* Lipsia: 2nd level from existing first level (runs from command line, results are in a different documnt)

In this document I take 3 types of first level contrasts and try 3 types of second level contrasts. <br>
For the t-test: I think that the most interesting contrast is 4back-1back, because it corresponds to added task complexity, but the effects of the working memory task cancell out. <br>
Additional information:

To try:
* different smooting
* MVPA
* fitting model
* nipy
* FSL

In [11]:
import sys
sys.path.append("..")
import os
import itertools
import argparse
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import nibabel as nib
import matplotlib as mpl
from nilearn import plotting
from nilearn import input_data, datasets
from nilearn.plotting import plot_stat_map, plot_anat, plot_img, show, plot_glass_brain, plot_connectome
from nilearn.glm.second_level import SecondLevelModel, non_parametric_inference
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.input_data import NiftiMasker
from nilearn.glm import threshold_stats_img, cluster_level_inference
from nilearn.image import get_data, math_img
from nilearn.reporting import make_glm_report
from scipy.stats import norm, t, ttest_rel

#%%Step 1: Loading data
data_dir = '/home/alado/datasets/RBH/preprocessed/derivatives/fmriprep'
top_dir = '/home/alado/datasets/RBH'
out_dir = '/home/alado/datasets/RBH/GLM/SLA/'
suffix = '_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz'
mask_suffix = '_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz'
sess = ['ses-1', 'ses-3']
task = 'nback'
t_r = 3.0

#contrast computed at the first level for each pre and post session, and then for mean pre-post
contrasts_fl = ['0back-1back','0back-2back','0back-4back','0back-fix',
             '1back-2back','1back-4back','1back-fix','2back-4back',
             '2back-fix','4back-fix','0back','1back','2back','4back','fix']

groups = pd.read_csv(f'{top_dir}/behavioural/subj_info.csv')
subs = pd.Series.tolist(groups['sub'][groups['group'].isin(['sport', 'med', 'control'])])
#add subjects with high motion in at least one session
#high_motion = ['sub-']
#high_motion_filter = ~subjects_data_clean['sub'].isin(high_motion).values
# Removing high-motion subjects from dataset
#subjects_data_clean_lm = subjects_data_clean[~subjects_data_clean['sub'].isin(high_motion)].reset_index()

I'll stast with simple t-test because it's easier to interpret

In [ ]:
# This script converts a T distribution brain map with dof
# degrees of freedom to a Z score map (standard normal) with
# mean 0, standard deviation 1. The algorithm is modified for
# brain images, and originally published here:
# http://www.stats.uwo.ca/faculty/aim/2010/JSSSnipets/V23N1.pdf

def t_to_z(tmap, dof):
    print ("Converting t map to Z-Scores...")
    
    # select the nonzero voxels
    nonzero = tmap[tmap!=0]
    
    # store the results here
    Z = np.zeros(len(nonzero))
    
    # Select values < or == 0, and > than zero
    c  = np.zeros(len(nonzero))
    k1 = (nonzero <= c)
    k2 = (nonzero > c)
    
    # Subset the data into two sets
    t1 = nonzero[k1]
    t2 = nonzero[k2]
    
    # Calculate p values for <=0
    p_values_t1 = t.cdf(t1, df = dof)
    z_values_t1 = norm.ppf(p_values_t1)
    
    # Calculate p values for > 0
    p_values_t2 = t.cdf(-t2, df = dof)
    z_values_t2 = -norm.ppf(p_values_t2)
    Z[k1] = z_values_t1
    Z[k2] = z_values_t2
    
    zmap = tmap
    zmap[tmap!=0] = Z
    
    return zmap

In [ ]:
#contr_fl_interest = ['4back-1back', '1back-4back', '4back-0back', '0back-4back', '2back-0back', '0back-2back']
contr_fl_interest = ['4back-1back']
subs_sport = pd.Series.tolist(groups['sub'][groups['group'].isin(['sport'])])
nsubs = len(subs_sport)

zmaps_s1 = [] 
for k, contrast in enumerate(contr_fl_interest):
    for i, sub in enumerate(subs_sport):
        zmap = nib.load(f'{top_dir}/GLM/FLA/voxel/{sub}_ses-1_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
        affine = zmap.affine
        zmap = zmap.get_fdata()
        zmaps_s1.append(zmap)
        
zmaps_s3 = [] 
for k, contrast in enumerate(contr_fl_interest):
    for i, sub in enumerate(subs_sport):
        zmap = nib.load(f'{top_dir}/GLM/FLA/voxel/{sub}_ses-3_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
        zmap = zmap.get_fdata()
        zmaps_s3.append(zmap)

t_val, p_val = ttest_rel(np.array(zmaps_s1), np.array(zmaps_s3), 0)
z_val = norm.isf(p_val)

#convert t to z
dof = nsubs - 2
#z_map = norm.ppf(t.cdf(t_val, dof)) #or t_val
z_map = t_to_z(t_val, dof)
z_map = nib.Nifti1Image(z_map, affine=affine)

thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
print('The FDR=.05 threshold is %.3g' % threshold2)

% Compute each contrast separately<br>
 Loading zmaps for contrasts defined like [1,-1,0,0,0]

In [ ]:
zmaps = [] 
for k, contrast in enumerate(contrasts_fl[0:16]):
    for j, ses in enumerate(sess):
        for i, sub in enumerate(subs):
            zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_{ses}_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
            
#%% Step 2: Create second-level design matrix (520)
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub * len(sess)
ses_arr = np.asarray([1,2])
ses_val = (ses_arr - np.mean(ses_arr))/np.std(ses_arr)
design_matrix['ses_pre'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(1,2)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(-1,1)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(2,0)
design_matrix['ses_post'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_post'] = design_matrix['ses_post'].replace(-1,0)
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]*2))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]*2))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]*2))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(np.concatenate([ds_sub, ds_sub]), columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
design_matrix = pd.concat([design_matrix]*16, ignore_index=True)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

#Step 3: Second level GLM analysis
second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)

# Step 4: estimate the contrast
n_columns = len(design_matrix.columns)
contrasts_sl = {'ses+control-sport': np.pad([0, -1, 1, 1, -1, 0], (0,n_columns-6)),
                'ses-control+sport': np.pad([0, -1, 1, -1, 1, 0], (0,n_columns-6)),
                'ses+control-med': np.pad([0, -1, 1, 1, 0, -1], (0,n_columns-6)),
                'ses-control+med': np.pad([0, -1, 1, -1, 0, 1], (0,n_columns-6)),
                'ses+sport-med': np.pad([0, -1, 1, 0, 1, -1], (0,n_columns-6)),
                'ses-sport+med': np.pad([0, -1, 1, 0, -1, 1], (0,n_columns-6))}

for index, (contrast_id, contrast_val) in enumerate(contrasts_sl.items()):
    z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
    
    #plot uncorrected map
    plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)
    
    #generate report html
    #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

    #%Step 5: Thresholded results
    #Compute the required threshold level and return the thresholded map (map, threshold)
    #add cluster_threshold
    _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    
    #plot the second level contrast at the computed thresholds
    #bg_img default
    plotting.plot_stat_map(
        z_map, threshold=threshold, colorbar=True,
        title='Group-level %s\n'
        '(fdr=0.01)' % contrast_id)
    plotting.show()

    #%Computing corrected p-values with parametric test to compare with non parametric test
    #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
    p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
    n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
    # Correcting the p-values for multiple testing and taking negative logarithm
    neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
    
    #plot the (corrected) negative log p-values for the parametric test
    
    # Since we are plotting negative log p-values and using a threshold equal to 1,
    # it corresponds to corrected p-values lower than 10%, meaning that there
    # is less than 10% probability to make a single false discovery
    # (90% chance that we make no false discoveries at all).
    # This threshold is much more conservative than the previous one.
    threshold = 1
    title = ('Group-level association between \n'
             'neg-log of parametric corrected p-values (FWER < 10%) \n
              %s', %contrast_id)
    plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                           threshold=threshold, title=title)
    plotting.show()
    
    #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
    thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    print('The FDR=.05 threshold is %.3g' % threshold2)
    
    #the fdr-thresholded map
    plotting.plot_stat_map(thresholded_map2,
                           title='Thresholded z map, expected fdr = .05 \n
                                  %s'%contrast_id,
                           threshold=threshold2
                           )
    plotting.show()

    #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
    #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
    
    thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
    print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
    
    #the Bonferroni-thresholded map
    plotting.plot_stat_map(thresholded_map3,
                           title='Thresholded z map, expected fwer < .05 \n
                                  %s'%contrast_id,
                           threshold=threshold3)
    plotting.show()

% Compute each contrast separately<br>
Loading zmaps for contrasts defined like [1,0,0,0,0]

In [ ]:
zmaps = []
for k, contrast in enumerate(contrasts_fl[10:]):
    for j, ses in enumerate(sess):
        for i, sub in enumerate(subs):
            zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_{ses}_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
                
#%Step 2: Create second-level design matrix
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub * len(sess)
ses_arr = np.asarray([1,2])
ses_val = (ses_arr - np.mean(ses_arr))/np.std(ses_arr)
design_matrix['ses_pre'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(1,2)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(-1,1)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(2,0)
design_matrix['ses_post'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_post'] = design_matrix['ses_post'].replace(-1,0)
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]*2))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]*2))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]*2))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(np.concatenate([ds_sub, ds_sub]), columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
design_matrix = pd.concat([design_matrix]*5, ignore_index=True)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

Step 3: Second level GLM analysis<br>
o smoothing on a previous step

In [ ]:
second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)

 Step 4: estimate the contrast (e.g. use the column name of the design matrix)

In [ ]:
n_columns = len(design_matrix.columns)
contrasts_sl = {'ses+control-sport': np.pad([0, -1, 1, 1, -1, 0], (0,n_columns-6)),
                'ses-control+sport': np.pad([0, -1, 1, -1, 1, 0], (0,n_columns-6)),
                'ses+control-med': np.pad([0, -1, 1, 1, 0, -1], (0,n_columns-6)),
                'ses-control+med': np.pad([0, -1, 1, -1, 0, 1], (0,n_columns-6)),
                'ses+sport-med': np.pad([0, -1, 1, 0, 1, -1], (0,n_columns-6)),
                'ses-sport+med': np.pad([0, -1, 1, 0, -1, 1], (0,n_columns-6))}

#Generate html report
make_glm_report(second_level_model, contrast).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report4_pos_fla.html')

In [ ]:
for index, (contrast_id, contrast_val) in enumerate(contrasts_sl.items()):
    z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
    
    #plot uncorrected map
    plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)
    
    #generate report html
    #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

    #%Step 5: Thresholded results
    #Compute the required threshold level and return the thresholded map (map, threshold)
    #add cluster_threshold
    _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    
    #plot the second level contrast at the computed thresholds
    #bg_img default
    plotting.plot_stat_map(
        z_map, threshold=threshold, colorbar=True,
        title='Group-level %s\n'
        '(fdr=0.01)' % contrast_id)
    plotting.show()

    #%Computing corrected p-values with parametric test to compare with non parametric test
    #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
    p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
    n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
    # Correcting the p-values for multiple testing and taking negative logarithm
    neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
    
    #plot the (corrected) negative log p-values for the parametric test
    
    # Since we are plotting negative log p-values and using a threshold equal to 1,
    # it corresponds to corrected p-values lower than 10%, meaning that there
    # is less than 10% probability to make a single false discovery
    # (90% chance that we make no false discoveries at all).
    # This threshold is much more conservative than the previous one.
    threshold = 1
    title = ('Group-level association between \n'
             'neg-log of parametric corrected p-values (FWER < 10%) \n
              %s', %contrast_id)
    plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                           threshold=threshold, title=title)
    plotting.show()
    
    #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
    thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    print('The FDR=.05 threshold is %.3g' % threshold2)
    
    #the fdr-thresholded map
    plotting.plot_stat_map(thresholded_map2,
                           title='Thresholded z map, expected fdr = .05 \n
                                  %s'%contrast_id,
                           threshold=threshold2
                           )
    plotting.show()

    #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
    #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
    
    thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
    print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
    
    #the Bonferroni-thresholded map
    plotting.plot_stat_map(thresholded_map3,
                           title='Thresholded z map, expected fwer < .05 \n
                                  %s'%contrast_id,
                           threshold=threshold3)
    plotting.show()

%each FL contrast separately<br>
 Loading zmaps for contrasts defined like [1,-1,0,0,0]<br>
Step 2: Create second-level design matrix

In [ ]:
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub * len(sess)
ses_arr = np.asarray([1,2])
ses_val = (ses_arr - np.mean(ses_arr))/np.std(ses_arr)
design_matrix['ses_pre'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(1,2)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(-1,1)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(2,0)
design_matrix['ses_post'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_post'] = design_matrix['ses_post'].replace(-1,0)
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]*2))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]*2))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]*2))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(np.concatenate([ds_sub, ds_sub]), columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)
    
n_columns = len(design_matrix.columns)
contrasts_sl = {'ses+control-sport': np.pad([0, -1, 1, 1, -1, 0], (0,n_columns-6)),
                'ses-control+sport': np.pad([0, -1, 1, -1, 1, 0], (0,n_columns-6)),
                'ses+control-med': np.pad([0, -1, 1, 1, 0, -1], (0,n_columns-6)),
                'ses-control+med': np.pad([0, -1, 1, -1, 0, 1], (0,n_columns-6)),
                'ses+sport-med': np.pad([0, -1, 1, 0, 1, -1], (0,n_columns-6)),
                'ses-sport+med': np.pad([0, -1, 1, 0, -1, 1], (0,n_columns-6))}

#Generate html report
make_glm_report(second_level_model, contrast).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report5_pos_and_neg_fla.html')

In [ ]:
for k, contrast in enumerate(contrasts_fl[0:10]):
    zmaps = []
    print('Contrast % 2i out of %i: %s' % (k + 1, len(contrasts_fl[0:10]), k))
    for j, ses in enumerate(sess):
        for i, sub in enumerate(subs):
            zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_{ses}_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
    
    #%Step 3: Second level GLM analysis 
    #no smoothing on a previous step
    second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
    second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)
    
    #% Step 4: estimate the contrast (e.g. use the column name of the design matrix)
    for index, (contrast_id, contrast_val) in enumerate(contrasts_sl.items()):
        z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
        #plot uncorrected map
        plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)

        #generate report html
        #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

        #%Step 5: Thresholded results
        #Compute the required threshold level and return the thresholded map (map, threshold)
        #add cluster_threshold
        _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')

        #plot the second level contrast at the computed thresholds
        #bg_img default
        plotting.plot_stat_map(
            z_map, threshold=threshold, colorbar=True,
            title='Group-level %s\n'
            '(fdr=0.01)' % contrast_id)
        plotting.show()

        #%Computing corrected p-values with parametric test to compare with non parametric test
        #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
        p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
        n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
        # Correcting the p-values for multiple testing and taking negative logarithm
        neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)

        #plot the (corrected) negative log p-values for the parametric test

        # Since we are plotting negative log p-values and using a threshold equal to 1,
        # it corresponds to corrected p-values lower than 10%, meaning that there
        # is less than 10% probability to make a single false discovery
        # (90% chance that we make no false discoveries at all).
        # This threshold is much more conservative than the previous one.
        threshold = 1
        title = ('Group-level association between \n'
                 'neg-log of parametric corrected p-values (FWER < 10%) \n
                  %s', %contrast_id)
        plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                               threshold=threshold, title=title)
        plotting.show()

        #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
        thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
        print('The FDR=.05 threshold is %.3g' % threshold2)

        #the fdr-thresholded map
        plotting.plot_stat_map(thresholded_map2,
                               title='Thresholded z map, expected fdr = .05 \n
                                      %s'%contrast_id,
                               threshold=threshold2
                               )
        plotting.show()

        #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
        #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.

        thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
        print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)

        #the Bonferroni-thresholded map
        plotting.plot_stat_map(thresholded_map3,
                               title='Thresholded z map, expected fwer < .05 \n
                                      %s'%contrast_id,
                               threshold=threshold3)
        plotting.show()

%each mean pre-post FL contrast separately<br>
Loading zmaps for contrasts defined like [1,0,0,0,0]<br>

In [ ]:
#Step 2: Create second-level design matrix
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub * len(sess)
ses_arr = np.asarray([1,2])
ses_val = (ses_arr - np.mean(ses_arr))/np.std(ses_arr)
design_matrix['ses_pre'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(1,2)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(-1,1)
design_matrix['ses_pre'] = design_matrix['ses_pre'].replace(2,0)
design_matrix['ses_post'] = list(itertools.chain.from_iterable([[i] * n_sub for i in ses_val]))
design_matrix['ses_post'] = design_matrix['ses_post'].replace(-1,0)
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]*2))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]*2))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]*2))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(np.concatenate([ds_sub, ds_sub]), columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)
    
n_columns = len(design_matrix.columns)
contrasts_sl = {'ses+control-sport': np.pad([0, -1, 1, 1, -1, 0], (0,n_columns-6)),
                'ses-control+sport': np.pad([0, -1, 1, -1, 1, 0], (0,n_columns-6)),
                'ses+control-med': np.pad([0, -1, 1, 1, 0, -1], (0,n_columns-6)),
                'ses-control+med': np.pad([0, -1, 1, -1, 0, 1], (0,n_columns-6)),
                'ses+sport-med': np.pad([0, -1, 1, 0, 1, -1], (0,n_columns-6)),
                'ses-sport+med': np.pad([0, -1, 1, 0, -1, 1], (0,n_columns-6))}


for k, contrast in enumerate(contrasts_fl[0:10]):
    zmaps = []
    print('Contrast % 2i out of %i: %s' % (k + 1, len(contrasts_fl[0:10]), k))
    for j, ses in enumerate(sess):
        for i, sub in enumerate(subs):
            zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_{ses}_space-MNI152NLin2009cAsym_{contrast}.nii.gz')

    #%Step 3: Second level GLM analysis
    #no smoothing on a previous step
    second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
    second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)
    
    #% Step 4: estimate the contrast (e.g. use the column name of the design matrix)
    for index, contrast_val in enumerate(contrasts_sl):
        z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
        #plot uncorrected map
        plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)

        #generate report html
        #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

        #%Step 5: Thresholded results
        #Compute the required threshold level and return the thresholded map (map, threshold)
        #add cluster_threshold
        _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')

        #plot the second level contrast at the computed thresholds
        #bg_img default
        plotting.plot_stat_map(
            z_map, threshold=threshold, colorbar=True,
            title='Group-level %s\n'
            '(fdr=0.01)' % contrast_id)
        plotting.show()

        #%Computing corrected p-values with parametric test to compare with non parametric test
        #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
        p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
        n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
        # Correcting the p-values for multiple testing and taking negative logarithm
        neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)

        #plot the (corrected) negative log p-values for the parametric test

        # Since we are plotting negative log p-values and using a threshold equal to 1,
        # it corresponds to corrected p-values lower than 10%, meaning that there
        # is less than 10% probability to make a single false discovery
        # (90% chance that we make no false discoveries at all).
        # This threshold is much more conservative than the previous one.
        threshold = 1
        title = ('Group-level association between \n'
                 'neg-log of parametric corrected p-values (FWER < 10%) \n
                  %s', %contrast_id)
        plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                               threshold=threshold, title=title)
        plotting.show()

        #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
        thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
        print('The FDR=.05 threshold is %.3g' % threshold2)

        #the fdr-thresholded map
        plotting.plot_stat_map(thresholded_map2,
                               title='Thresholded z map, expected fdr = .05 \n
                                      %s'%contrast_id,
                               threshold=threshold2
                               )
        plotting.show()

        #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
        #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.

        thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
        print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)

        #the Bonferroni-thresholded map
        plotting.plot_stat_map(thresholded_map3,
                               title='Thresholded z map, expected fwer < .05 \n
                                      %s'%contrast_id,
                               threshold=threshold3)
        plotting.show()

%Compute from mean pre-post<br>
 Loading zmaps for contrasts defined like [1,-1,0,0,0]

In [ ]:
zmaps = [] 
for k, contrast in enumerate(contrasts_fl[0:10]):
        for i, sub in enumerate(subs):
            zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_mean_space-MNI152NLin2009cAsym_{contrast}.nii.gz')

#Step 2: Create second-level design matrix (520)
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(ds_sub, columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
design_matrix = pd.concat([design_matrix]*10, ignore_index=True)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

#Step 3: Second level GLM analysis
second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)

#Step 4: estimate the contrast
n_columns = len(design_matrix.columns)
contrasts_sl = {'control-sport': np.pad([0, 1, -1, 0], (0,n_columns-4)),
                'control+sport': np.pad([0, -1, 1, 0], (0,n_columns-4)),
                'control-med': np.pad([0, 1, 0, -1], (0,n_columns-4)),
                'control+med': np.pad([0, -1, 0, 1], (0,n_columns-4)),
                'sport-med': np.pad([0, 0, 1, -1], (0,n_columns-4)),
                'sport+med': np.pad([0, 0, -1, 1], (0,n_columns-4))}

for index, (contrast_id, contrast_val) in enumerate(contrasts_sl.items()):
    z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
    
    #plot uncorrected map
    plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)
    
    #generate report html
    #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

    #%Step 5: Thresholded results
    #Compute the required threshold level and return the thresholded map (map, threshold)
    #add cluster_threshold
    _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    
    #plot the second level contrast at the computed thresholds
    #bg_img default
    plotting.plot_stat_map(
        z_map, threshold=threshold, colorbar=True,
        title='Group-level %s\n'
        '(fdr=0.01)' % contrast_id)
    plotting.show()

    #%Computing corrected p-values with parametric test to compare with non parametric test
    #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
    p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
    n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
    # Correcting the p-values for multiple testing and taking negative logarithm
    neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
    
    #plot the (corrected) negative log p-values for the parametric test
    
    # Since we are plotting negative log p-values and using a threshold equal to 1,
    # it corresponds to corrected p-values lower than 10%, meaning that there
    # is less than 10% probability to make a single false discovery
    # (90% chance that we make no false discoveries at all).
    # This threshold is much more conservative than the previous one.
    threshold = 1
    title = ('Group-level association between \n'
             'neg-log of parametric corrected p-values (FWER < 10%) \n
              %s', %contrast_id)
    plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                           threshold=threshold, title=title)
    plotting.show()
    
    #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
    thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    print('The FDR=.05 threshold is %.3g' % threshold2)
    
    #the fdr-thresholded map
    plotting.plot_stat_map(thresholded_map2,
                           title='Thresholded z map, expected fdr = .05 \n
                                  %s'%contrast_id,
                           threshold=threshold2
                           )
    plotting.show()

    #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
    #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
    
    thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
    print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
    
    #the Bonferroni-thresholded map
    plotting.plot_stat_map(thresholded_map3,
                           title='Thresholded z map, expected fwer < .05 \n
                                  %s'%contrast_id,
                           threshold=threshold3)
    plotting.show()

%Compute from mean pre-post<br>
Loading zmaos for contrasts defined like [1,0,0,0,0]

In [ ]:
zmaps = []
for k, contrast in enumerate(contrasts_fl[10:]):
        for i, sub in enumerate(subs):
            zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_mean_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
                
#%Step 2: Create second-level design matrix
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(ds_sub, columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
design_matrix = pd.concat([design_matrix]*5, ignore_index=True)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

#%%Step 3: Second level GLM analysis
second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)

#Step 4: estimate the contrast (e.g. use the column name of the design matrix)
n_columns = len(design_matrix.columns)
contrasts_sl = {'control-sport': np.pad([0, 1, -1, 0], (0,n_columns-4)),
                'control+sport': np.pad([0, -1, 1, 0], (0,n_columns-4)),
                'control-med': np.pad([0, 1, 0, -1], (0,n_columns-4)),
                'control+med': np.pad([0, -1, 0, 1], (0,n_columns-4)),
                'sport-med': np.pad([0, 0, 1, -1], (0,n_columns-4)),
                'sport+med': np.pad([0, 0, -1, 1], (0,n_columns-4))}

for index, (contrast_id, contrast_val) in enumerate(contrasts_sl.items()):
    z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
    
    #plot uncorrected map
    plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)
    
    #generate report html
    #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

    #%Step 5: Thresholded results
    #Compute the required threshold level and return the thresholded map (map, threshold)
    #add cluster_threshold
    _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    
    #plot the second level contrast at the computed thresholds
    #bg_img default
    plotting.plot_stat_map(
        z_map, threshold=threshold, colorbar=True,
        title='Group-level %s\n'
        '(fdr=0.01)' % contrast_id)
    plotting.show()

    #%Computing corrected p-values with parametric test to compare with non parametric test
    #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
    p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
    n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
    # Correcting the p-values for multiple testing and taking negative logarithm
    neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
    
    #plot the (corrected) negative log p-values for the parametric test
    
    # Since we are plotting negative log p-values and using a threshold equal to 1,
    # it corresponds to corrected p-values lower than 10%, meaning that there
    # is less than 10% probability to make a single false discovery
    # (90% chance that we make no false discoveries at all).
    # This threshold is much more conservative than the previous one.
    threshold = 1
    title = ('Group-level association between \n'
             'neg-log of parametric corrected p-values (FWER < 10%) \n
              %s', %contrast_id)
    plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                           threshold=threshold, title=title)
    plotting.show()
    
    #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
    thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    print('The FDR=.05 threshold is %.3g' % threshold2)
    
    #the fdr-thresholded map
    plotting.plot_stat_map(thresholded_map2,
                           title='Thresholded z map, expected fdr = .05 \n
                                  %s'%contrast_id,
                           threshold=threshold2
                           )
    plotting.show()

    #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
    #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
    
    thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
    print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
    
    #the Bonferroni-thresholded map
    plotting.plot_stat_map(thresholded_map3,
                           title='Thresholded z map, expected fwer < .05 \n
                                  %s'%contrast_id,
                           threshold=threshold3)
    plotting.show()

%compute contrast as column in design matrix from pre-post files<br>
 Loading zmaps for contrasts defined like [1,-1,0,0,0]

In [ ]:
zmaps = [] 
for k, contrast in enumerate(contrasts_fl[0:10]):
    for i, sub in enumerate(subs):
        zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_mean_space-MNI152NLin2009cAsym_{contrast}.nii.gz')

#Step 2: Create second-level design matrix (520)
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(ds_sub, columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
design_matrix = pd.concat([design_matrix]*10, ignore_index=True)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

#Step 3: Second level GLM analysis
second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)

#Step 4: estimate the contrast (use the column name of the design matrix)
contrasts_sl = ['control', 'sport', 'med']

for index, contrast_id in enumerate(contrasts_sl):
    z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
    
    #plot uncorrected map
    plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)
    
    #generate report html
    #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

    #%Step 5: Thresholded results
    #Compute the required threshold level and return the thresholded map (map, threshold)
    #add cluster_threshold
    _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    
    #plot the second level contrast at the computed thresholds
    #bg_img default
    plotting.plot_stat_map(
        z_map, threshold=threshold, colorbar=True,
        title='Group-level %s\n'
        '(fdr=0.01)' % contrast_id)
    plotting.show()

    #%Computing corrected p-values with parametric test to compare with non parametric test
    #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
    p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
    n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
    # Correcting the p-values for multiple testing and taking negative logarithm
    neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
    
    #plot the (corrected) negative log p-values for the parametric test
    
    # Since we are plotting negative log p-values and using a threshold equal to 1,
    # it corresponds to corrected p-values lower than 10%, meaning that there
    # is less than 10% probability to make a single false discovery
    # (90% chance that we make no false discoveries at all).
    # This threshold is much more conservative than the previous one.
    threshold = 1
    title = ('Group-level association between \n'
             'neg-log of parametric corrected p-values (FWER < 10%) \n
              %s', %contrast_id)
    plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                           threshold=threshold, title=title)
    plotting.show()
    
    #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
    thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    print('The FDR=.05 threshold is %.3g' % threshold2)
    
    #the fdr-thresholded map
    plotting.plot_stat_map(thresholded_map2,
                           title='Thresholded z map, expected fdr = .05 \n
                                  %s'%contrast_id,
                           threshold=threshold2
                           )
    plotting.show()

    #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
    #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
    
    thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
    print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
    
    #the Bonferroni-thresholded map
    plotting.plot_stat_map(thresholded_map3,
                           title='Thresholded z map, expected fwer < .05 \n
                                  %s'%contrast_id,
                           threshold=threshold3)
    plotting.show()
        
    #%Computing the (corrected) negative log p-values with permutation test
    
    neg_log_pvals_permuted_ols_unmasked = non_parametric_inference(zmaps,
                                  design_matrix=design_matrix,
                                  second_level_contrast=contrast_val,
                                  model_intercept=True, n_perm=1000,
                                  two_sided_test=False, mask=None,
                                  smoothing_fwhm=5.0, n_jobs=1)
        
    #plot the (corrected) negative log p-values
    
    title = ('Group-level association between \n'
              'neg-log of non-parametric corrected p-values (FWER < 10%)')
    plotting.plot_stat_map(neg_log_pvals_permuted_ols_unmasked, colorbar=True,
                            #cut_coords=cut_coords,
                            threshold=threshold,
                            title=title)
    plotting.show()
    
    # The neg-log p-values obtained with non parametric testing are capped at 3
    # since the number of permutations is 1e3.
    # The non parametric test yields a few more discoveries
    # and is then more powerful than the usual parametric procedure.

%compute contrast as column in design matrix from pre-post files<br>
Loading zmaos for contrasts defined like [1,0,0,0,0]

In [ ]:
zmaps = []
for k, contrast in enumerate(contrasts_fl[10:]):
    for i, sub in enumerate(subs):
        zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_mean_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
                
#%Step 2: Create second-level design matrix
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]))
ds_sub = np.eye(len(groups['sub']))
ds_subs = pd.DataFrame(ds_sub, columns = groups['sub'])
design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
design_matrix = pd.concat([design_matrix]*5, ignore_index=True)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

#Step 3: Second level GLM analysis
second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)

#Step 4: estimate the contrast (use the column name of the design matrix)
contrasts_sl = ['control', 'sport', 'med']

for index, contrast_id in enumerate(contrasts_sl):
    z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
    
    #plot uncorrected map
    plotting.plot_stat_map(z_map, threshold=2.0,title='uncorrected %s' % contrast_id)
    
    #generate report html
    #make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report3.html')

    #%Step 5: Thresholded results
    #Compute the required threshold level and return the thresholded map (map, threshold)
    #add cluster_threshold
    _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    
    #plot the second level contrast at the computed thresholds
    #bg_img default
    plotting.plot_stat_map(
        z_map, threshold=threshold, colorbar=True,
        title='Group-level %s\n'
        '(fdr=0.01)' % contrast_id)
    plotting.show()

    #%Computing corrected p-values with parametric test to compare with non parametric test
    #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
    p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
    n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
    # Correcting the p-values for multiple testing and taking negative logarithm
    neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
    
    #plot the (corrected) negative log p-values for the parametric test
    
    # Since we are plotting negative log p-values and using a threshold equal to 1,
    # it corresponds to corrected p-values lower than 10%, meaning that there
    # is less than 10% probability to make a single false discovery
    # (90% chance that we make no false discoveries at all).
    # This threshold is much more conservative than the previous one.
    threshold = 1
    title = ('Group-level association between \n'
             'neg-log of parametric corrected p-values (FWER < 10%) \n
              %s', %contrast_id)
    plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                           threshold=threshold, title=title)
    plotting.show()
    
    #%FDR <.05 (False Discovery Rate) and no cluster-level threshold
    thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
    print('The FDR=.05 threshold is %.3g' % threshold2)
    
    #the fdr-thresholded map
    plotting.plot_stat_map(thresholded_map2,
                           title='Thresholded z map, expected fdr = .05 \n
                                  %s'%contrast_id,
                           threshold=threshold2
                           )
    plotting.show()

    #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
    #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
    
    thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
    print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
    
    #the Bonferroni-thresholded map
    plotting.plot_stat_map(thresholded_map3,
                           title='Thresholded z map, expected fwer < .05 \n
                                  %s'%contrast_id,
                           threshold=threshold3)
    plotting.show()
        
    #%Computing the (corrected) negative log p-values with permutation test
    
    neg_log_pvals_permuted_ols_unmasked = non_parametric_inference(zmaps,
                                  design_matrix=design_matrix,
                                  second_level_contrast=contrast_val,
                                  model_intercept=True, n_perm=1000,
                                  two_sided_test=False, mask=None,
                                  smoothing_fwhm=5.0, n_jobs=1)
        
    #plot the (corrected) negative log p-values
    
    title = ('Group-level association between \n'
              'neg-log of non-parametric corrected p-values (FWER < 10%)')
    plotting.plot_stat_map(neg_log_pvals_permuted_ols_unmasked, colorbar=True,
                            #cut_coords=cut_coords,
                            threshold=threshold,
                            title=title)
    plotting.show()
    
    # The neg-log p-values obtained with non parametric testing are capped at 3
    # since the number of permutations is 1e3.
    # The non parametric test yields a few more discoveries
    # and is then more powerful than the usual parametric procedure.

%each mean pre-post FL contrast separately<br>
 Loading zmaps for contrasts defined like [1,-1,0,0,0]<br>
Step 2: Create second-level design matrix

In [ ]:
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]))
#ds_sub = np.eye(len(groups['sub']))
#ds_subs = pd.DataFrame(ds_sub, columns = groups['sub'])
#design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)
    
contrasts_sl = ['control', 'sport', 'med']

for k, contrast in enumerate(contrasts_fl[0:10]):
    zmaps = []
    print('Contrast % 2i out of %i: %s' % (k + 1, len(contrasts_fl[0:10]), k))
    for i, sub in enumerate(subs):
        zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_mean_space-MNI152NLin2009cAsym_{contrast}.nii.gz')
    
    #%Step 3: Second level GLM analysis 
    #no smoothing on a previous step
    second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
    second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)
    
    #% Step 4: estimate the contrast (e.g. use the column name of the design matrix)
    for index, contrast_val in enumerate(contrasts_sl):
        z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
        plotting.plot_stat_map(z_map, threshold=2.0, title='%s' % contrast_val)
        
        #generate report html
        make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/repor11_pos_and_neg_fla.html')
    
        #%Step 5: FDR-thresholded result
        #Compute the required threshold level and return the thresholded map (map, threshold)
        #add cluster_threshold
        _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
        
        #plot the second level contrast at the computed thresholds
        #bg_img default
        plotting.plot_stat_map(
            z_map, threshold=threshold, colorbar=True,
            title='Group-level %s\n'
            '(fdr=0.01)' % contrast_val)
        plotting.show()
    
        #%Computing corrected p-values with parametric test to compare with non parametric test
        #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
        p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
        n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
        # Correcting the p-values for multiple testing and taking negative logarithm
        neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
        
        #plot the (corrected) negative log p-values for the parametric test
        
        #cut_coords = [50, -17, -3]
        # Since we are plotting negative log p-values and using a threshold equal to 1,
        # it corresponds to corrected p-values lower than 10%, meaning that there
        # is less than 10% probability to make a single false discovery
        # (90% chance that we make no false discoveries at all).
        # This threshold is much more conservative than the previous one.
        threshold = 1
        title = ('Group-level association between \n'
                 'neg-log of parametric corrected p-values (FWER < 10%)')
        plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                               #cut_coords=cut_coords,
                               threshold=threshold, title=title)
        plotting.show()
    
        #%threshold the second level contrast at uncorrected p < 0.001 and plot
        p_val = 0.001
        p001_uncorrected = norm.isf(p_val) #Inverse survival function
        
        proportion_true_discoveries_img = cluster_level_inference(z_map, threshold=[3, 4, 5], alpha=.05)
        
        plotting.plot_stat_map(
            proportion_true_discoveries_img, threshold=0., colorbar=True, display_mode='z',
            title='group, proportion true positives', vmax=1)
        
        plotting.plot_stat_map(
            z_map, threshold=p001_uncorrected, colorbar=True, display_mode='z',
            title='group (uncorrected p < 0.001)')
        plotting.show()
    
        
        #%threshold the second level contrast and plot it
        
        threshold = 3.1  # correponds to  p < .001, uncorrected
        display = plotting.plot_glass_brain(
            z_map, threshold=threshold, colorbar=True, plot_abs=False,
            title='vertical vs horizontal checkerboard (unc p<0.001')
        plotting.show()
    
        #%Threshold the resulting map: false positive rate < .001, cluster size > 10 voxels
        
        thresholded_map1, threshold1 = threshold_stats_img(z_map, alpha=.001, height_control='fpr', cluster_threshold=10)
        
        #p<.001 uncorrected-thresholded map (with only clusters > 10 voxels).
        
        plotting.plot_stat_map(
            z_map,
            #thresholded_map1, 
            #cut_coords=display.cut_coords, 
            threshold=threshold1,
            title='Thresholded z map, fpr <.001, clusters > 10 voxels')
        plotting.show()
        
        #%FDR <.05 (False Discovery Rate) and no cluster-level threshold.
        
        thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
        print('The FDR=.05 threshold is %.3g' % threshold2)
        
        #the fdr-thresholded map
        plotting.plot_stat_map(thresholded_map2,
                               #cut_coords=display.cut_coords,
                               title='Thresholded z map, expected fdr = .05',
                               threshold=threshold2
                               )
        plotting.show()
    
        #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
        #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
        
        thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
        print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
        
        #the Bonferroni-thresholded map
        
        plotting.plot_stat_map(thresholded_map3,
                               #cut_coords=display.cut_coords,
                               title='Thresholded z map, expected fwer < .05',
                               threshold=threshold3)
        plotting.show()
        
        #%Computing the (corrected) negative log p-values with permutation test
        
        neg_log_pvals_permuted_ols_unmasked = non_parametric_inference(zmaps,
                                      design_matrix=design_matrix,
                                      second_level_contrast=contrast_val,
                                      model_intercept=True, n_perm=1000,
                                      two_sided_test=False, mask=None,
                                      smoothing_fwhm=5.0, n_jobs=1)
            
        #plot the (corrected) negative log p-values
        
        title = ('Group-level association between \n'
                  'neg-log of non-parametric corrected p-values (FWER < 10%)')
        plotting.plot_stat_map(neg_log_pvals_permuted_ols_unmasked, colorbar=True,
                                #cut_coords=cut_coords,
                                threshold=threshold,
                                title=title)
        plotting.show()
        
        # The neg-log p-values obtained with non parametric testing are capped at 3
        # since the number of permutations is 1e3.
        # The non parametric test yields a few more discoveries
        # and is then more powerful than the usual parametric procedure.

%each mean pre-post FL contrast separately<br>
Loading zmaos for contrasts defined like [1,0,0,0,0]<br>
Step 2: Create second-level design matrix

In [ ]:
n_sub = len(subs)
design_matrix = pd.DataFrame()
design_matrix['constant'] = [1] * n_sub
design_matrix['control'] = list(itertools.chain.from_iterable([(groups['group']=='control').values.astype(int)]))
design_matrix['sport'] = list(itertools.chain.from_iterable([(groups['group']=='sport').values.astype(int)]))
design_matrix['med'] = list(itertools.chain.from_iterable([(groups['group']=='med').values.astype(int)]))
#ds_sub = np.eye(len(groups['sub']))
#ds_subs = pd.DataFrame(np.concatenate([ds_sub, ds_sub]), columns = groups['sub'])
#design_matrix = pd.concat((design_matrix, ds_subs), axis=1)
#plot
ax = plot_design_matrix(design_matrix)
ax.get_images()[0].set_clim(0, 0.2)

contrasts_sl = ['control', 'sport', 'med']

In [ ]:
for k, contrast in enumerate(contrasts_fl[10:]):
    zmaps = []
    for i, sub in enumerate(subs):
        zmaps.append(f'{top_dir}/GLM/FLA/voxel/{sub}_mean_space-MNI152NLin2009cAsym_{contrast}.nii.gz')

    #%Step 3: Second level GLM analysis
    #no smoothing on a previous step
    second_level_model = SecondLevelModel(smoothing_fwhm=6.0)
    second_level_model = second_level_model.fit(zmaps, design_matrix=design_matrix)
    
    #% Step 4: estimate the contrast (e.g. use the column name of the design matrix)
    for index, contrast_val in enumerate(contrasts_sl):
        z_map = second_level_model.compute_contrast(contrast_val, output_type='z_score')
        plotting.plot_stat_map(z_map, threshold=2.0,title='%s' % contrast_id)
        
        #generate report html
        make_glm_report(second_level_model, contrast_val).save_as_html('/home/alado/datasets/RBH/GLM/SLA/voxel/report12_pos_fla.html')
    
        #%Step 5: FDR-thresholded result
        #Compute the required threshold level and return the thresholded map (map, threshold)
        #add cluster_threshold
        _, threshold = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
        
        #plot the second level contrast at the computed thresholds
        #bg_img default
        plotting.plot_stat_map(
            z_map, threshold=threshold, colorbar=True,
            title='Group-level %s\n'
            '(fdr=0.01)' % contrast_id)
        plotting.show()
    
        #%Computing corrected p-values with parametric test to compare with non parametric test
        #gives RuntimeWarning: divide by zero encountered in log10, but it's ok
        p_val = second_level_model.compute_contrast(contrast_val, output_type='p_value')
        n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
        # Correcting the p-values for multiple testing and taking negative logarithm
        neg_log_pval = math_img("-np.log10(np.minimum(1, img * {}))".format(str(n_voxels)),img=p_val)
        
        #plot the (corrected) negative log p-values for the parametric test
        
        #cut_coords = [50, -17, -3]
        # Since we are plotting negative log p-values and using a threshold equal to 1,
        # it corresponds to corrected p-values lower than 10%, meaning that there
        # is less than 10% probability to make a single false discovery
        # (90% chance that we make no false discoveries at all).
        # This threshold is much more conservative than the previous one.
        threshold = 1
        title = ('Group-level association between \n'
                 'neg-log of parametric corrected p-values (FWER < 10%)')
        plotting.plot_stat_map(neg_log_pval, colorbar=True, 
                               #cut_coords=cut_coords,
                               threshold=threshold, title=title)
        plotting.show()
    
        #%threshold the second level contrast at uncorrected p < 0.001 and plot
        p_val = 0.001
        p001_uncorrected = norm.isf(p_val) #Inverse survival function
        
        proportion_true_discoveries_img = cluster_level_inference(z_map, threshold=[3, 4, 5], alpha=.05)
        
        plotting.plot_stat_map(
            proportion_true_discoveries_img, threshold=0., colorbar=True, display_mode='z',
            title='group, proportion true positives', vmax=1)
        
        plotting.plot_stat_map(
            z_map, threshold=p001_uncorrected, colorbar=True, display_mode='z',
            title='group (uncorrected p < 0.001)')
        plotting.show()
    
        
        #%threshold the second level contrast and plot it
        
        threshold = 3.1  # correponds to  p < .001, uncorrected
        display = plotting.plot_glass_brain(
            z_map, threshold=threshold, colorbar=True, plot_abs=False,
            title='vertical vs horizontal checkerboard (unc p<0.001')
        plotting.show()
    
        #%Threshold the resulting map: false positive rate < .001, cluster size > 10 voxels
        
        thresholded_map1, threshold1 = threshold_stats_img(z_map, alpha=.001, height_control='fpr', cluster_threshold=10)
        
        #p<.001 uncorrected-thresholded map (with only clusters > 10 voxels).
        
        plotting.plot_stat_map(
            z_map,
            #thresholded_map1, 
            #cut_coords=display.cut_coords, 
            threshold=threshold1,
            title='Thresholded z map, fpr <.001, clusters > 10 voxels')
        plotting.show()
        
        #%FDR <.05 (False Discovery Rate) and no cluster-level threshold.
        
        thresholded_map2, threshold2 = threshold_stats_img(z_map, alpha=.05, height_control='fdr')
        print('The FDR=.05 threshold is %.3g' % threshold2)
        
        #the fdr-thresholded map
        plotting.plot_stat_map(thresholded_map2,
                               #cut_coords=display.cut_coords,
                               title='Thresholded z map, expected fdr = .05',
                               threshold=threshold2
                               )
        plotting.show()
    
        #%Now use FWER <.05 (Family-Wise Error Rate) and no cluster-level threshold.
        #If the data has not been intensively smoothed, we can use a simple Bonferroni correction.
        
        thresholded_map3, threshold3 = threshold_stats_img(z_map, alpha=.05, height_control='bonferroni')
        print('The p<.05 Bonferroni-corrected threshold is %.3g' % threshold3)
        
        #the Bonferroni-thresholded map
        
        plotting.plot_stat_map(thresholded_map3,
                               #cut_coords=display.cut_coords,
                               title='Thresholded z map, expected fwer < .05',
                               threshold=threshold3)
        plotting.show()
        
        #%Computing the (corrected) negative log p-values with permutation test
        
        neg_log_pvals_permuted_ols_unmasked = non_parametric_inference(zmaps,
                                      design_matrix=design_matrix,
                                      second_level_contrast=contrast_val,
                                      model_intercept=True, n_perm=1000,
                                      two_sided_test=False, mask=None,
                                      smoothing_fwhm=5.0, n_jobs=1)
            
        #plot the (corrected) negative log p-values
        
        title = ('Group-level association between \n'
                  'neg-log of non-parametric corrected p-values (FWER < 10%)')
        plotting.plot_stat_map(neg_log_pvals_permuted_ols_unmasked, colorbar=True,
                                #cut_coords=cut_coords,
                                threshold=threshold,
                                title=title)
        plotting.show()
        
        # The neg-log p-values obtained with non parametric testing are capped at 3
        # since the number of permutations is 1e3.
        # The non parametric test yields a few more discoveries
        # and is then more powerful than the usual parametric procedure.